<a href="https://colab.research.google.com/github/vv2281/MAT-422/blob/main/HW_3_4_VV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MAT 422 — Section 3.4: Logistic Regression


1. Model and sigmoid function
2. Likelihood and cross-entropy loss
3. Gradient, Hessian, convexity
4. Gradient descent and Newton's method
5. Decision boundary and odds ratios
6. Separable data and regularization
7. Real-data example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(422)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

##1. The Model

Response yᵢ ∈ {0, 1}, features xᵢ ∈ Rᵈ (first entry 1 for the intercept). Linear regression can predict values outside [0, 1]...

So the linear score is passed through the sigmoid:

P(yᵢ = 1 | xᵢ) = σ(xᵢᵀβ),   σ(z) = 1 / (1 + e^(−z))

Equivalently, the log-odds are linear in x:

log[ p / (1 − p) ] = xᵀβ

Sigmoid properties used below:

1. 0 < σ(z) < 1,   σ(0) = ½
2. σ(−z) = 1 − σ(z)
3. σ'(z) = σ(z)(1 − σ(z))

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 400)
h = 1e-6
print("max |σ(−z) − (1 − σ(z))|        =", np.abs(sigmoid(-z) - (1 - sigmoid(z))).max())
print("max |σ'(z) − σ(z)(1 − σ(z))|     =", np.abs((sigmoid(z + h) - sigmoid(z - h)) / (2 * h) - sigmoid(z) * (1 - sigmoid(z))).max())
print("log-odds of σ(z) equals z:       ", np.allclose(np.log(sigmoid(z) / (1 - sigmoid(z))), z))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(z, sigmoid(z))
ax[0].axhline(0.5, color="gray", ls=":")
ax[0].set_title("σ(z)"); ax[0].set_xlabel("z")
ax[1].plot(z, sigmoid(z) * (1 - sigmoid(z)))
ax[1].set_title("σ'(z) = σ(z)(1 − σ(z)), max ¼ at z = 0"); ax[1].set_xlabel("z")
plt.tight_layout(); plt.show()

##2. Likelihood and Cross-Entropy Loss

Each yᵢ ~ Bernoulli(pᵢ) with pᵢ = σ(xᵢᵀβ). As in the Bernoulli MLE of Section 2.4, the log-likelihood is

ℓ(β) = ∑ [ yᵢ log pᵢ + (1 − yᵢ) log(1 − pᵢ) ]

There is no closed-form maximizer. Maximizing ℓ is the same as minimizing the average negative log-likelihood, the cross-entropy loss:

f(β) = −(1/n) ∑ [ yᵢ log pᵢ + (1 − yᵢ) log(1 − pᵢ) ]

##3. Gradient, Hessian, Convexity

Using σ' = σ(1 − σ):

∇f(β) = (1/n) Xᵀ(p − y)
H(β)  = (1/n) XᵀWX,   W = diag(pᵢ(1 − pᵢ))

For any v: vᵀHv = (1/n) ∑ pᵢ(1 − pᵢ)(xᵢᵀv)² ≥ 0, so H is PSD and f is convex (Section 3.3). Any β with ∇f(β) = 0 is a global minimizer.

Since pᵢ(1 − pᵢ) ≤ ¼, every Hessian eigenvalue is at most

L = λmax(XᵀX) / (4n)

...so gradient descent with α = 1/L decreases f every step.

##Example 1: Simulated data

Two features, true β = (−1, 2, −3). The gradient formula is checked against finite differences, and the Hessian eigenvalues are checked to be nonnegative at random β.

In [ ]:
n = 300
X_raw = rng.normal(size=(n, 2))
X = np.column_stack([np.ones(n), X_raw])
beta_true = np.array([-1.0, 2.0, -3.0])
y = rng.binomial(1, sigmoid(X @ beta_true))

def loss(beta, X, y):
    p = np.clip(sigmoid(X @ beta), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def grad(beta, X, y):
    return X.T @ (sigmoid(X @ beta) - y) / len(y)

def hess(beta, X):
    p = sigmoid(X @ beta)
    return X.T @ (X * (p * (1 - p))[:, None]) / len(X)

b0 = rng.normal(size=3)
num_g = np.array([(loss(b0 + 1e-6 * e, X, y) - loss(b0 - 1e-6 * e, X, y)) / 2e-6 for e in np.eye(3)])
print("analytic gradient :", np.round(grad(b0, X, y), 6))
print("numerical gradient:", np.round(num_g, 6))
min_eigs = [np.linalg.eigvalsh(hess(rng.normal(scale=3, size=3), X)).min() for _ in range(1000)]
print("smallest Hessian eigenvalue over 1000 random β:", round(min(min_eigs), 6))

plt.scatter(X_raw[y == 0, 0], X_raw[y == 0, 1], alpha=0.6, label="y = 0")
plt.scatter(X_raw[y == 1, 0], X_raw[y == 1, 1], alpha=0.6, label="y = 1")
plt.xlabel("x₁"); plt.ylabel("x₂"); plt.title("Simulated data")
plt.legend(); plt.show()

##4. Fitting the Model

Gradient descent:

βₖ₊₁ = βₖ − α ∇f(βₖ),   α = 1/L

Newton's method uses the Hessian to rescale the step:

βₖ₊₁ = βₖ − H(βₖ)⁻¹ ∇f(βₖ)

Newton's method needs far fewer iterations (quadratic convergence near the minimum), but each step solves a d × d linear system. Both are compared with scikit-learn (C = 1e10 makes its penalty negligible).

In [ ]:
L = np.linalg.eigvalsh(X.T @ X).max() / (4 * n)
beta_gd = np.zeros(3)
gd_loss = [loss(beta_gd, X, y)]
for _ in range(500):
    beta_gd = beta_gd - (1 / L) * grad(beta_gd, X, y)
    gd_loss.append(loss(beta_gd, X, y))

beta_nt = np.zeros(3)
nt_loss = [loss(beta_nt, X, y)]
for _ in range(10):
    beta_nt = beta_nt - np.linalg.solve(hess(beta_nt, X), grad(beta_nt, X, y))
    nt_loss.append(loss(beta_nt, X, y))

sk = LogisticRegression(C=1e10, max_iter=10000, tol=1e-10).fit(X_raw, y)
beta_sk = np.concatenate([sk.intercept_, sk.coef_.ravel()])

print(f"L = {L:.4f}")
print(f"{'':22s}{'β₀':>10s}{'β₁':>10s}{'β₂':>10s}")
for name, b in [("true", beta_true), ("GD (500 steps)", beta_gd), ("Newton (10 steps)", beta_nt), ("scikit-learn", beta_sk)]:
    print(f"{name:22s}{b[0]:>10.4f}{b[1]:>10.4f}{b[2]:>10.4f}")
print("‖∇f‖ at Newton solution:", np.linalg.norm(grad(beta_nt, X, y)))

f_min = nt_loss[-1]
plt.semilogy(np.array(gd_loss) - f_min + 1e-16, label="gradient descent")
plt.semilogy(np.array(nt_loss) - f_min + 1e-16, "o-", label="Newton")
plt.xlabel("iteration"); plt.ylabel("f(βₖ) − f*")
plt.title("Convergence"); plt.legend(); plt.show()

##5. Decision Boundary and Odds Ratios

Classify ŷ = 1 when p̂ ≥ ½, which happens exactly when xᵀβ̂ ≥ 0. The decision boundary (β₀ + β₁x₁ + β₂x₂ = 0) is a line (a hyperplane in higher dimensions), so logistic regression is a linear classifier.

Interpretation: increasing xⱼ by 1 with the other features fixed adds βⱼ to the log-odds, so it multiplies the odds by: e^(βⱼ)   (the odds ratio)

In [ ]:
p_hat = sigmoid(X @ beta_nt)
y_hat = (p_hat >= 0.5).astype(int)
tp, tn = np.sum((y_hat == 1) & (y == 1)), np.sum((y_hat == 0) & (y == 0))
fp, fn = np.sum((y_hat == 1) & (y == 0)), np.sum((y_hat == 0) & (y == 1))
print(f"accuracy = {(tp + tn) / n:.3f}")
print(f"confusion matrix [[TN FP] [FN TP]] = [[{tn} {fp}] [{fn} {tp}]]")

x_pt = np.array([1.0, 0.3, -0.2])
odds = lambda v: sigmoid(v @ beta_nt) / (1 - sigmoid(v @ beta_nt))
print(f"odds(x₁ + 1) / odds(x₁) = {odds(x_pt + [0, 1, 0]) / odds(x_pt):.4f},  e^β₁ = {np.exp(beta_nt[1]):.4f}")

g = np.linspace(-3.5, 3.5, 300)
G1, G2 = np.meshgrid(g, g)
P = sigmoid(beta_nt[0] + beta_nt[1] * G1 + beta_nt[2] * G2)
plt.figure(figsize=(6.5, 5))
cf = plt.contourf(G1, G2, P, levels=20, cmap="RdBu_r", alpha=0.6)
plt.colorbar(cf, label="p̂ = P(y = 1 | x)")
plt.contour(G1, G2, P, levels=[0.5], colors="k", linewidths=2)
plt.scatter(X_raw[y == 0, 0], X_raw[y == 0, 1], c="blue", s=12, label="y = 0")
plt.scatter(X_raw[y == 1, 0], X_raw[y == 1, 1], c="red", s=12, label="y = 1")
plt.xlabel("x₁"); plt.ylabel("x₂"); plt.title("Fitted probabilities; black line = boundary p̂ = ½")
plt.legend(); plt.show()

##6. Separable Data and Regularization

If a line separates the two classes perfectly, scaling β up makes every pᵢ closer to the correct label, so f(cβ) → 0 as c → ∞. The infimum is never attained: the MLE does not exist, and gradient descent sends ‖β‖ → ∞.

L2 regularization (ridge) fixes this: f_λ(β) = f(β) + (λ/2)‖β‖²

Its Hessian is H + λI, with every eigenvalue ≥ λ > 0, so f_λ is strongly convex and has a unique minimizer. (The intercept is usually left unpenalized.)

In [ ]:
Xs_raw = np.vstack([rng.normal([-2, -2], 0.7, size=(50, 2)), rng.normal([2, 2], 0.7, size=(50, 2))])
Xs = np.column_stack([np.ones(100), Xs_raw])
ys = np.repeat([0, 1], 50)
Ls = np.linalg.eigvalsh(Xs.T @ Xs).max() / (4 * 100)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for lam in [0.0, 0.01, 0.1]:
    b = np.zeros(3)
    norms, losses = [], []
    for _ in range(20000):
        b = b - (1 / (Ls + lam)) * (grad(b, Xs, ys) + lam * b)
        norms.append(np.linalg.norm(b))
        losses.append(loss(b, Xs, ys))
    print(f"λ = {lam:<5} ‖β‖ after 20000 steps = {norms[-1]:8.3f},  cross-entropy = {losses[-1]:.2e}")
    ax[0].plot(norms, label=f"λ = {lam}")
    ax[1].semilogy(losses, label=f"λ = {lam}")
ax[0].set_xlabel("iteration"); ax[0].set_title("‖βₖ‖"); ax[0].legend()
ax[1].set_xlabel("iteration"); ax[1].set_title("cross-entropy f(βₖ)"); ax[1].legend()
plt.tight_layout(); plt.show()

With λ = 0, ‖β‖ keeps growing (roughly like log k) while the loss creeps toward 0. With λ > 0, β converges to a finite point.

##7. Real Data: Breast Cancer Diagnosis

569 tumors, 30 features, y = 1 if benign. Features are standardized (mean 0, SD 1) so gradient descent is well conditioned and coefficients are comparable.

scikit-learn's default minimizes ∑ cross-entropy + (1/(2C))‖w‖² with C = 1. Dividing by n, that is the regularized loss above with λ = 1/(Cn) on the non-intercept coefficients, so our Newton fit should match it exactly.

In [ ]:
data = load_breast_cancer()
Xtr_raw, Xte_raw, ytr, yte = train_test_split(data.data, data.target, test_size=0.25, random_state=0, stratify=data.target)
mu, sd = Xtr_raw.mean(axis=0), Xtr_raw.std(axis=0)
Xtr = np.column_stack([np.ones(len(ytr)), (Xtr_raw - mu) / sd])
Xte = np.column_stack([np.ones(len(yte)), (Xte_raw - mu) / sd])

lam = 1 / len(ytr)
D = np.eye(Xtr.shape[1]); D[0, 0] = 0
beta = np.zeros(Xtr.shape[1])
for _ in range(25):
    g_reg = grad(beta, Xtr, ytr) + lam * D @ beta
    beta = beta - np.linalg.solve(hess(beta, Xtr) + lam * D, g_reg)

sk = LogisticRegression(C=1.0, max_iter=10000, tol=1e-10).fit(Xtr[:, 1:], ytr)
beta_sk = np.concatenate([sk.intercept_, sk.coef_.ravel()])
acc = np.mean((sigmoid(Xte @ beta) >= 0.5) == yte)
print("‖∇f_λ‖ at solution:", np.linalg.norm(grad(beta, Xtr, ytr) + lam * D @ beta))
print("max |β − β_sklearn| :", np.abs(beta - beta_sk).max())
print(f"test accuracy       : {acc:.3f}   (scikit-learn: {sk.score(Xte[:, 1:], yte):.3f})")

order = np.argsort(np.abs(beta[1:]))[-10:]
plt.figure(figsize=(7, 4.5))
plt.barh(np.array(data.feature_names)[order], beta[1:][order])
plt.xlabel("coefficient (per 1 SD increase)")
plt.title("10 largest coefficients (negative = lower odds of benign)")
plt.tight_layout(); plt.show()

#Conclusion:

- Model: P(y = 1 | x) = σ(xᵀβ); the log-odds are linear in x.
- Fitting is maximum likelihood for Bernoulli data, i.e. minimizing cross-entropy f(β).
- ∇f = (1/n)Xᵀ(p − y), H = (1/n)XᵀWX is PSD, so f is convex and ∇f = 0 gives a global minimum.
- No closed form: use gradient descent (α = 1/L, L = λmax(XᵀX)/(4n)) or Newton's method (few iterations, quadratic convergence).
- The decision boundary xᵀβ = 0 is linear; e^(βⱼ) is the odds ratio for a 1-unit increase in xⱼ.
- For separable data the MLE does not exist; L2 regularization makes the loss strongly convex with a unique minimizer.